For the first steps of the pipeline I will be using popglen, so I need to format raw data to match requirements. 




#Preparing the files 
First I do the samples.tsv

In [14]:
cd /media/ssd/Raw_data/Picoides/Nuclear_data_02_2025/fastq_by_species/dorsalis #file directory 

ls *.fastq.gz | cut -d'_' -f1-2 | sort | uniq > dorsalis_seqnames.txt #create a list of samples 

# match the names in sample_name_key (this is a metadata file) to the list of samples
while read name; do     grep "$name" /media/ssd/Raw_data/Picoides/Nuclear_data_02_2025/Sample_name_key.txt; 
done < dorsalis_seqnames.txt > dorsalis_key_matches.txt

#create a population table 
awk -F',' 'NR>1 {print $1"\t"$8}' \
/media/ssd/Raw_data/Picoides/Nuclear_data_02_2025/woodpeckers_metadata.csv \
> population_key.txt

#join together the Ids and the population 
awk '
NR==FNR {pop[$1]=$2; next}
{print $1"\t"$2"\t"pop[$1]}
' population_key.txt dorsalis_key_matches.txt > dorsalis_joined.txt

# get sample id and population from woodpecker_metadata # since all samples are modern and low I specify it manually
awk '{print $1"\t"$3"\tmodern\tlow"}' dorsalis_joined.txt > samples_dorsalis.tsv

#add column names
sed -i '1i sample\tpopulation\ttime\tdepth' samples_dorsalis.tsv



In [5]:
#populations have weird names so I need to standardize all names
awk '{print $2}' samples_dorsalis.tsv | sort -u ##extract all unique population names

#fix the population names
awk '
{
    gsub(/Ê/, "", $2)

    if ($2 == "British") $2 = "BritishColumbia"
    if ($2 == "New") $2 = "NewMexico"

    print
}
' OFS="\t" samples_dorsalis.tsv > samples_fixed.tsv
awk '{print $2}' samples_fixed.tsv | sort -u ##check if it worked

# replace the temporary samples_fixed since it worked
mv samples_fixed.tsv samples_dorsalis.tsv

Alaska
Alberta
Arizona
BritishColumbia
Colorado
Manitoba
NewMexico
Oregon
population
Quebec
Utah
Washington


Now I need to make the units.tsv file 

In [10]:
echo -e "sample_id\tunit\tlibrary\tplatform\tfq1\tfq2" > units.tsv

for fq1 in *_R1_*.fastq.gz; do
    fq2=${fq1/_R1_/_R2_}

    filename=$(basename "$fq1")

    library=$(cut -d'_' -f1 <<< "$filename")
    unit=$(cut -d'_' -f2 <<< "$filename")

    key="${library}_${unit}"

    sample_id=$(awk -v key="$key" '$2==key {print $1}' \
        /media/ssd/Raw_data/Picoides/Nuclear_data_02_2025/Sample_name_key.txt)

    if [[ -z "$sample_id" ]]; then
        echo "WARNING: no sample_id for $key" >&2
        continue
    fi

    echo -e "${sample_id}\t${unit}\t${library}\tILLUMINA\t$(realpath "$fq1")\t$(realpath "$fq2")" >> units.tsv
done

#echo $(($(ls *.fastq.gz | wc -l) / 2)) #check that the number of files in folder match the units.tsv file 

60
